# Recurrent Neural Networks

Pichhle module mein humne text ke rich semantic representations cover kiye the. Jo architecture hum use kar rahe the, woh sentence mein words ka overall meaning capture karta hai, lekin yeh words ke **order** ko dhyan mein nahi rakhta, kyunki embedding ke baad ka aggregation operation original text se yeh information hata deta hai. Kyunki yeh models words ka order represent nahi kar paate, isliye woh text generation ya question answering jaise zyada complex ya ambiguous tasks solve nahi kar sakte.

Text sequence ka meaning capture karne ke liye hum ek neural network architecture use karenge jise **Recurrent Neural Network** ya RNN kehte hain. RNN use karte waqt hum apna sentence network ke through ek-ek token karke pass karte hain, aur network kuch **state** produce karta hai, jise hum agle token ke saath wapas network mein pass kar dete hain.

![Recurrent Neural Network generation ka ek example dikhane wali image.](../../../../../lessons/5-NLP/16-RNN/images/rnn.png)

Token sequence $X_0,\dots,X_n$ ke liye, RNN neural network blocks ki ek sequence banata hai, aur is sequence ko backpropagation se end-to-end train karta hai. Har network block $(X_i,S_i)$ ki ek pair ko input ke roop mein leta hai, aur result mein $S_{i+1}$ produce karta hai. Final state $S_n$ ya output $Y_n$ ek linear classifier mein jaata hai jisse result milta hai. Saare network blocks same weights share karte hain, aur ek backpropagation pass se end-to-end train hote hain.

> Upar di gayi figure recurrent neural network ko unrolled form (left side) mein aur zyada compact recurrent representation (right side) mein dikhati hai. Yeh samajhna zaroori hai ki saare RNN cells ke **shareable weights** same hote hain.

Kyunki state vectors $S_0,\dots,S_n$ network ke through pass hote hain, RNN words ke beech sequential dependencies seekh sakta hai. Jaise, jab sequence mein kahin *not* word aata hai, toh yeh state vector ke kuch elements ko negate karna seekh sakta hai.

Andar se, har RNN cell mein do weight matrices hote hain: $W_H$ aur $W_I$, aur ek bias $b$. Har RNN step par, input $X_i$ aur input state $S_i$ diye jaane par, output state is tarah calculate hoti hai: $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, jahan $f$ ek activation function hai (aksar $\tanh$).

> Text generation (jise hum agli unit mein cover karenge) ya machine translation jaisi problems ke liye, hum har RNN step par kuch output value bhi chahte hain. Is case mein ek aur matrix $W_O$ hota hai, aur output is tarah calculate hota hai: $Y_i=f(W_O\times S_i+b_O)$.

Chalo dekhte hain ki recurrent neural networks hamare news dataset ko classify karne mein kaise madad kar sakte hain.

> Sandbox environment ke liye, humein neeche wala cell run karna hoga taaki yeh pakka ho jaye ki zaroori library install ho gayi hai, aur data prefetch ho gaya hai. Agar aap locally run kar rahe hain, toh aap neeche wala cell skip kar sakte hain.

In [ ]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [ ]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Bade models ko train karte waqt GPU memory allocation ek problem ban sakti hai. Humein alag-alag minibatch sizes ke saath experiment karna pad sakta hai, taaki data hamari GPU memory mein fit ho jaye aur training kaafi fast rahe. Agar aap yeh code apni GPU machine par run kar rahe hain, toh training tez karne ke liye minibatch size adjust karke experiment kar sakte hain.

> **Note**: NVidia drivers ke kuch versions ke baare mein pata hai ki woh model ki training khatam hone ke baad memory release nahi karte. Hum is notebook mein kai examples run kar rahe hain, aur isse kuch setups mein memory khatam ho sakti hai, khaaskar agar aap isi notebook mein apne khud ke experiments kar rahe hain. Agar model training shuru karte waqt aapko kuch ajeeb errors dikhein, toh aap notebook kernel ko restart karna chahenge.

In [ ]:
batch_size = 16
embed_size = 64

## Simple RNN classifier

Simple RNN ke case mein, har recurrent unit ek simple linear network hota hai, jo ek input vector aur ek state vector leta hai, aur ek naya state vector produce karta hai. Keras mein ise `SimpleRNN` layer se represent kiya ja sakta hai.

Haalaanki hum RNN layer ko seedhe one-hot encoded tokens pass kar sakte hain, yeh ek achha idea nahi hai kyunki unki dimensionality bahut high hoti hai. Isliye hum word vector ki dimensionality kam karne ke liye ek embedding layer use karenge, uske baad ek RNN layer, aur end mein ek `Dense` classifier.

> **Note**: Jin cases mein dimensionality itni high nahi hoti, jaise jab character-level tokenization use hoti hai, wahan one-hot encoded tokens ko seedhe RNN cell mein pass karna samajhdari ho sakti hai.

In [ ]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

> **Dhyan dein:** Yahan hum simplicity ke liye ek untrained embedding layer use kar rahe hain, lekin behtar results ke liye hum Word2Vec se pre-trained embedding layer use kar sakte hain, jaisa pichhli unit mein bataya gaya tha. Aapke liye yeh ek achhi practice hogi ki aap is code ko pre-trained embeddings ke saath kaam karne ke liye adapt karein.

Ab chalo apne RNN ko train karte hain. Generally RNN ko train karna kaafi mushkil hota hai, kyunki jab RNN cells ko sequence length ke saath unroll kiya jaata hai, toh backpropagation mein shamil layers ki ginti bahut zyada ho jaati hai. Isliye humein chhota learning rate chunna padta hai, aur achhe results paane ke liye network ko bade dataset par train karna padta hai. Isme kaafi time lag sakta hai, isliye GPU use karna behtar hota hai.

Process ko tez karne ke liye, hum RNN model ko sirf news titles par train karenge aur description ko chhod denge. Aap description ke saath training try kar sakte hain aur dekh sakte hain ki model train ho paata hai ya nahi.

In [ ]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

In [ ]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

> **Note** ki accuracy yahan kam ho sakti hai, kyunki hum sirf news titles par training kar rahe hain.

## Variable sequences ke baare mein dobara sochna

Yaad rakhiye ki `TextVectorization` layer ek minibatch mein variable length wali sequences ko automatically pad tokens se pad kar deti hai. Dekha gaya hai ki yeh tokens bhi training mein hissa lete hain, aur woh model ki convergence ko complicated bana sakte hain.

Padding ki matra kam karne ke liye hum kai tareeke use kar sakte hain. Inme se ek hai dataset ko sequence length ke hisaab se reorder karna aur saari sequences ko unke size ke hisaab se group karna. Yeh `tf.data.experimental.bucket_by_sequence_length` function se kiya ja sakta hai (dekhein [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)).

Ek aur tareeka **masking** use karna hai. Keras mein, kuch layers extra input support karti hain jo dikhata hai ki training ke dauraan kin tokens ko dhyan mein rakhna chahiye. Apne model mein masking shamil karne ke liye, hum ya toh alag se ek `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)) add kar sakte hain, ya hum apni `Embedding` layer mein `mask_zero=True` parameter specify kar sakte hain.

> **Note**: Is training mein poore dataset par ek epoch complete karne mein lagbhag 5 minute lagenge. Agar aapka sabr toot jaye toh training ko kisi bhi waqt rokne ke liye swatantra rahiye. Aap yeh bhi kar sakte hain ki training ke liye use hone wale data ki matra limit kar dein, `ds_train` aur `ds_test` datasets ke baad `.take(...)` clause add karke.

In [ ]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

Ab jab hum masking use kar rahe hain, toh hum titles aur descriptions ke poore dataset par model ko train kar sakte hain.

> **Note**: Kya aapne dhyan diya ki hum woh vectorizer use kar rahe hain jo news titles par train hua tha, na ki article ki poori body par? Ho sakta hai isse kuch tokens unseen reh jayein, isliye vectorizer ko dobara train karna behtar hoga. Lekin, iska effect bahut chhota ho sakta hai, isliye hum simplicity ke liye pehle se trained vectorizer hi use karte rahenge.

## LSTM: Long Short-Term Memory

RNNs ki main problems mein se ek hai **vanishing gradients**. RNNs kaafi lambe ho sakte hain, aur backpropagation ke dauraan gradients ko network ki pehli layer tak poori tarah wapas le jaana mushkil ho sakta hai. Jab aisa hota hai, toh network door-door ke tokens ke beech relationships seekhne mein asamarth ho jaata hai. Is problem se bachne ka ek tareeka hai **explicit state management** ko **gates** use karke implement karna. Gates introduce karne wale do sabse common architectures hain **Long Short Term Memory** (LSTM) aur **Gated Relay Unit** (GRU). Yahan hum LSTMs cover karenge.

![Long short-term memory cell ka ek example dikhane wali image](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Ek LSTM network ko RNN ki tarah hi organize kiya jaata hai, lekin isme do states hoti hain jo layer se layer pass hoti hain: actual state $c$, aur hidden vector $h$. Har unit par, hidden vector $h_{t-1}$ ko input $x_t$ ke saath combine kiya jaata hai, aur yeh dono milkar **gates** ke through state $c_t$ aur output $h_{t}$ ko control karte hain. Har gate mein sigmoid activation hoti hai (output $[0,1]$ ki range mein), jise state vector se multiply karne par bitwise mask ki tarah socha ja sakta hai. LSTMs mein yeh gates hote hain (upar di gayi picture mein left se right):
* **forget gate** jo decide karta hai ki vector $c_{t-1}$ ke kaun se components humein bhoolne hain, aur kaun se aage pass karne hain.
* **input gate** jo decide karta hai ki input vector aur previous hidden vector se kitni information state vector mein shamil karni chahiye.
* **output gate** jo nayi state vector leta hai aur decide karta hai ki uske kaun se components naya hidden vector $h_t$ banane mein use honge.

State $c$ ke components ko aise flags ki tarah socha ja sakta hai jinhe on aur off kiya ja sakta hai. Jaise, jab hum sequence mein naam *Alice* se milte hain, toh hum andaza lagate hain ki yeh ek female ko refer karta hai, aur state mein woh flag raise kar dete hain jo kehta hai ki sentence mein humare paas ek feminine noun hai. Jab aage hum *and Tom* words se milte hain, toh hum woh flag raise karte hain jo kehta hai ki humare paas ek plural noun hai. Is tarah state ko manipulate karke hum sentence ki grammatical properties ka track rakh sakte hain.

> **Note**: LSTMs ke internal structure ko samajhne ke liye yeh ek shaandaar resource hai: Christopher Olah ka [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/).

Haalaanki LSTM cell ka internal structure complex lag sakta hai, Keras is implementation ko `LSTM` layer ke andar chhupa deta hai, isliye upar diye gaye example mein humein bas recurrent layer badalni hoti hai:

In [ ]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

## Bidirectional aur multilayer RNNs

Ab tak ke hamare examples mein, recurrent networks sequence ki shuruaat se end tak kaam karte hain. Yeh humein natural lagta hai kyunki yeh usi direction ko follow karta hai jisme hum padhte ya speech sunte hain. Lekin, un situations ke liye jahan input sequence ka random access zaroori ho, dono directions mein recurrent computation chalana zyada samajhdari hoti hai. Aise RNNs jo dono directions mein computation allow karte hain, unhe **bidirectional** RNNs kehte hain, aur inhe ek special `Bidirectional` layer se recurrent layer ko wrap karke banaya ja sakta hai.

> **Note**: `Bidirectional` layer apni andar wali layer ki do copies banati hai aur unme se ek ki `go_backwards` property ko `True` set kar deti hai, jisse woh sequence ke saath ulti direction mein jaati hai.

Recurrent networks, chahe ek direction mein ho ya bidirectional, sequence ke andar ke patterns ko capture karte hain aur unhe state vectors mein store karte hain ya output ke roop mein return karte hain. Jaise convolutional networks mein hota hai, hum pehli layer dwara nikaale gaye low-level patterns se high-level patterns capture karne ke liye pehli layer ke baad ek aur recurrent layer bana sakte hain. Isse hum **multilayer RNN** ke concept tak pahunchte hain, jisme do ya zyada recurrent networks hote hain, jahan pichhli layer ka output agli layer ko input ke roop mein diya jaata hai.

![Multilayer long-short-term-memory RNN dikhane wali image](../../../../../lessons/5-NLP/16-RNN/images/multi-layer-lstm.jpg)

*Fernando Lopez ki [is shaandaar post](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) se li gayi picture.*

Keras in networks ko banana aasaan bana deta hai, kyunki aapko bas model mein aur recurrent layers add karni hoti hain. Aakhri layer ko chhodkar baaki saari layers ke liye, humein `return_sequences=True` parameter specify karna hota hai, kyunki humein layer se saari intermediate states chahiye, na ki sirf recurrent computation ki final state.

Chalo apni classification problem ke liye ek two-layer bidirectional LSTM banate hain.

> **Note** Yeh code phir se complete hone mein kaafi time leta hai, lekin yeh humein ab tak ki sabse zyada accuracy deta hai. Toh shayad intezaar karna aur results dekhna sahi rahega.

In [ ]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

## Dusre tasks ke liye RNNs

Ab tak humne RNNs ko text sequences classify karne ke liye use kiya hai. Lekin woh aur bhi kai tasks sambhal sakte hain, jaise text generation aur machine translation — in tasks par hum agli unit mein vichaar karenge.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->